# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

print("Secret successfully loaded!")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    'dim_clients':f"read_parquet('{rel}/dim_clients.parquet')",
    'dim_content':f"read_parquet('{rel}/dim_content.parquet')",
    'fact_content_daily_performance':f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    'fact_content_query_90d':f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}


Secret successfully loaded!


In [3]:
df = con.sql(f"""
WITH content_id_hv_criteria AS (
    SELECT
        content_hash_id
    FROM {TABLES['fact_content_daily_performance']}
    WHERE
        gsc_data_available IS TRUE
        AND ga4_data_available IS NOT NULL
    GROUP BY content_hash_id
    HAVING
        COUNT(DISTINCT report_date) >= 120
        AND
        DATE_DIFF(
            'day',
            MIN(report_date),
            MAX(report_date)
        ) + 1 = COUNT(DISTINCT report_date)
)

SELECT fact.*
FROM {TABLES['fact_content_daily_performance']} AS fact
INNER JOIN content_id_hv_criteria AS criteria
    ON fact.content_hash_id = criteria.content_hash_id
WHERE
    fact.gsc_data_available IS TRUE
    AND fact.ga4_data_available IS NOT NULL;
""").df()

df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-02-10,client_9958f0a7ae1df715,content_92fd77e367253da2,True,True,True,False,4,0,29,...,0,0,0,0,0,0,0,0,0,2025-02
1,2025-02-10,client_9958f0a7ae1df715,content_00dafe96928080fb,True,True,True,False,18,0,88,...,0,0,0,0,0,0,0,0,0,2025-02
2,2025-02-10,client_9958f0a7ae1df715,content_7a1bdd56e533fef5,True,True,True,False,14,0,122,...,0,0,0,0,0,0,0,0,0,2025-02
3,2025-02-10,client_9958f0a7ae1df715,content_d73205f6d972a269,True,True,True,False,33,3,127,...,0,0,0,0,0,0,0,0,0,2025-02
4,2025-02-10,client_9958f0a7ae1df715,content_c936b727ae06f104,True,True,True,False,18,0,258,...,0,0,0,0,0,0,0,0,0,2025-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562564,2026-06-30,client_65de48885f4ef01b,content_f862eedcac7d187e,True,True,True,False,25,0,201,...,0,0,0,0,0,0,0,0,0,2026-06
2562565,2026-06-30,client_65de48885f4ef01b,content_153da632dd6adb12,True,True,True,False,44,0,355,...,0,0,0,0,0,0,0,0,0,2026-06
2562566,2026-06-30,client_65de48885f4ef01b,content_4e0c7263aa6aab62,True,True,True,False,63,1,350,...,0,0,0,0,0,0,0,0,0,2026-06
2562567,2026-06-30,client_65de48885f4ef01b,content_fa9131f1743ae0ae,True,True,True,False,73,0,376,...,0,0,0,0,0,0,0,0,0,2026-06


In [43]:
result = df.groupby('content_hash_id').agg(start_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['start_date','end_date'], ascending=[False,True])
result

,start_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_9987be00b8c14a02,2025-01-27,2026-06-30
content_cf0500bbb61cc8b9,2025-01-27,2026-06-30
content_d02be57d816cf3d7,2025-01-27,2026-06-30


In [50]:
import pandas as pd

In [57]:
incomplete_content_month = result[
    (result['start_date'].dt.day > 15) |
    (result['end_date'].dt.day < 15)
].copy().reset_index()

incomplete_content_month['start_month'] = (
    incomplete_content_month['start_date'].dt.to_period('M')
)

incomplete_content_month['end_month'] = (
    incomplete_content_month['end_date'].dt.to_period('M')
)

df['month'] = df['report_date'].dt.to_period('M')

data_complete_month = df.merge(
    incomplete_content_month[
        ['content_hash_id', 'start_month', 'end_month']
    ],
    on='content_hash_id',
    how='left'
)

data_complete_month = data_complete_month[
    (data_complete_month['month'] != data_complete_month['start_month']) &
    (data_complete_month['month'] != data_complete_month['end_month'])
].copy()

In [58]:
data_complete_month

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,start_month,end_month
0,2025-02-10,client_9958f0a7ae1df715,content_92fd77e367253da2,True,True,True,False,4,0,29,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
1,2025-02-10,client_9958f0a7ae1df715,content_00dafe96928080fb,True,True,True,False,18,0,88,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
2,2025-02-10,client_9958f0a7ae1df715,content_7a1bdd56e533fef5,True,True,True,False,14,0,122,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
3,2025-02-10,client_9958f0a7ae1df715,content_d73205f6d972a269,True,True,True,False,33,3,127,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
4,2025-02-10,client_9958f0a7ae1df715,content_c936b727ae06f104,True,True,True,False,18,0,258,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562499,2026-06-30,client_0fa64a184f18a4a0,content_36ce16237926c3eb,True,True,True,False,12,0,277,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562544,2026-06-30,client_9958f0a7ae1df715,content_00d12fab87754f25,True,True,True,False,6,0,415,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562545,2026-06-30,client_9958f0a7ae1df715,content_b2b445e21988683c,True,True,True,False,10,0,85,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562546,2026-06-30,client_9958f0a7ae1df715,content_8bea9167e438fad6,True,True,True,False,12,0,131,...,0,0,0,0,0,0,0,2026-06,NaT,NaT


In [62]:
data_complete_month.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2352188 entries, 0 to 2562547
Data columns (total 33 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   report_date               datetime64[us]
 1   client_hash_id            object        
 2   content_hash_id           object        
 3   client_has_gsc            bool          
 4   client_has_ga4            bool          
 5   gsc_data_available        bool          
 6   ga4_data_available        bool          
 7   gsc_impressions           int64         
 8   gsc_clicks                int64         
 9   gsc_sum_position          int64         
 10  gsc_avg_position          float64       
 11  ga4_pageviews             int64         
 12  ga4_sessions              int64         
 13  ga4_users                 int64         
 14  ga4_engaged_sessions      int64         
 15  ga4_total_engagement_sec  int64         
 16  sessions_organic          int64         
 17  sessions_dire

In [63]:
data_complete_month

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,start_month,end_month
0,2025-02-10,client_9958f0a7ae1df715,content_92fd77e367253da2,True,True,True,False,4,0,29,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
1,2025-02-10,client_9958f0a7ae1df715,content_00dafe96928080fb,True,True,True,False,18,0,88,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
2,2025-02-10,client_9958f0a7ae1df715,content_7a1bdd56e533fef5,True,True,True,False,14,0,122,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
3,2025-02-10,client_9958f0a7ae1df715,content_d73205f6d972a269,True,True,True,False,33,3,127,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
4,2025-02-10,client_9958f0a7ae1df715,content_c936b727ae06f104,True,True,True,False,18,0,258,...,0,0,0,0,0,0,0,2025-02,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2562499,2026-06-30,client_0fa64a184f18a4a0,content_36ce16237926c3eb,True,True,True,False,12,0,277,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562544,2026-06-30,client_9958f0a7ae1df715,content_00d12fab87754f25,True,True,True,False,6,0,415,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562545,2026-06-30,client_9958f0a7ae1df715,content_b2b445e21988683c,True,True,True,False,10,0,85,...,0,0,0,0,0,0,0,2026-06,NaT,NaT
2562546,2026-06-30,client_9958f0a7ae1df715,content_8bea9167e438fad6,True,True,True,False,12,0,131,...,0,0,0,0,0,0,0,2026-06,NaT,NaT


In [65]:
data_complete_month.groupby('content_hash_id').agg(start_date = ('report_date', "count")).sort_values('start_date',ascending=True)

,start_date
content_hash_id,
content_c6cb59c4632f7d0f,91
content_004047a006c4edc5,92
content_003959d7e5063269,92
content_016d125d80d9d257,92
content_a6fa750db5dc546c,92
...,...
content_e6802fc89ce19d51,512
content_00d12fab87754f25,513
content_b34ba5f684d53d57,513


In [66]:
data_complete_month.groupby('content_hash_id').agg(start_date = ('report_date', "min"),end_date = ('report_date', "max")).sort_values(['start_date','end_date'], ascending=[False,True])

,start_date,end_date
content_hash_id,,
content_0060ca245492494f,2026-03-03,2026-06-30
content_00823406ee06d1e7,2026-03-03,2026-06-30
content_0084ee8efa2fcd23,2026-03-03,2026-06-30
content_00e0da358d5df2e7,2026-03-03,2026-06-30
content_00efcc9c1c9bba3d,2026-03-03,2026-06-30
...,...,...
content_9987be00b8c14a02,2025-02-01,2026-05-31
content_cf0500bbb61cc8b9,2025-02-01,2026-05-31
content_d02be57d816cf3d7,2025-02-01,2026-05-31


In [67]:
data_complete_month[data_complete_month['content_hash_id']=='content_c6cb59c4632f7d0f']

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,start_month,end_month
5762,2025-05-28,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,11,0,167,...,0,0,0,0,0,0,0,2025-05,2025-03,2025-07
11443,2025-04-03,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,6,0,129,...,0,0,0,0,0,0,0,2025-04,2025-03,2025-07
11457,2025-04-10,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,11,0,243,...,0,0,0,0,0,0,0,2025-04,2025-03,2025-07
11477,2025-04-20,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,8,0,168,...,0,0,0,0,0,0,0,2025-04,2025-03,2025-07
11482,2025-04-28,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,24,0,423,...,0,0,0,0,0,0,0,2025-04,2025-03,2025-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1271735,2025-06-04,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,15,0,349,...,0,0,0,0,0,0,0,2025-06,2025-03,2025-07
1304533,2025-06-10,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,12,0,493,...,0,0,0,0,0,0,0,2025-06,2025-03,2025-07
1304612,2025-06-18,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,12,0,194,...,0,0,0,0,0,0,0,2025-06,2025-03,2025-07
1304622,2025-06-07,client_73cda7b4e4f265ea,content_c6cb59c4632f7d0f,True,True,True,False,18,0,225,...,0,0,0,0,0,0,0,2025-06,2025-03,2025-07


In [10]:
grouped = df.groupby([ 'client_hash_id','content_hash_id', 'month'])

numeric_cols = df.select_dtypes(include='number').columns.drop('gsc_avg_position')
bool_cols = df.select_dtypes(include='bool').columns

content_month = grouped.agg({
    **{col: 'sum' for col in numeric_cols},
    **{col: 'any' for col in bool_cols},
    'gsc_avg_position':'mean'
}).reset_index()

In [11]:
content_month

,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_avg_position
0,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-02,111,0,843,0,0,0,0,...,0,0,0,0,0,True,True,True,False,7.557559
1,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-03,330,1,2543,0,0,0,0,...,0,0,0,0,0,True,True,True,False,7.675690
2,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-04,393,5,2817,5,5,4,0,...,0,0,0,0,0,True,True,True,True,7.441862
3,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-05,872,1,6082,1,1,1,0,...,0,0,0,0,0,True,True,True,True,7.100545
4,client_0e1acc6cd57b0eba,content_2abf26b922d62f2b,2026-06,1437,6,8742,8,8,8,1,...,0,0,0,0,1,True,True,True,True,6.115233
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91227,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-02,939,4,8060,5,5,5,0,...,0,0,0,0,0,True,True,True,True,8.860769
91228,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-03,1534,3,13546,6,6,6,0,...,0,0,0,0,0,True,True,True,True,9.573120
91229,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-04,482,0,9770,4,4,3,0,...,0,0,0,0,0,True,True,True,True,20.424837
91230,client_ff644d8251367cbb,content_f3ba8c1f94eb36f5,2026-05,269,0,8335,5,5,5,1,...,0,0,0,0,1,True,True,True,True,32.566968


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.